# 🐍 Day 2: Decorators Advanced

- Decorators with arguments
- Stacking decorators
- functools.wraps
- Class-based decorators

## Decorators with Arguments

When you need to pass parameters to a decorator, use a factory: a function that returns the actual decorator.

In [ ]:
def repeat(times):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def say_hi():
    print("Hi!")
    return "done"

say_hi()

## Optional Arguments in Decorators

Support both `@decorator` and `@decorator()`. Check if the first argument is a function.

In [ ]:
def optional_repeat(func=None, *, times=2):
    def decorator(f):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = f(*args, **kwargs)
            return result
        return wrapper

    if func is not None:
        return decorator(func)
    return decorator

@optional_repeat
def greet1():
    print("Hello")
    return 1

@optional_repeat(times=3)
def greet2():
    print("Hi")
    return 2

greet1()
print("---")
greet2()

## Stacking Decorators

Decorators are applied bottom-to-top. The innermost decorator wraps the function first.

In [ ]:
def bold(func):
    def wrapper(*args, **kwargs):
        return f"**{func(*args, **kwargs)}**"
    return wrapper

def italic(func):
    def wrapper(*args, **kwargs):
        return f"*{func(*args, **kwargs)}*"
    return wrapper

@bold
@italic
def greet():
    return "Hello"

# Same as: greet = bold(italic(greet))
# italic wraps greet, bold wraps italic's wrapper
print(greet())  # **\*Hello\***

## functools.wraps

Without `wraps`, the wrapper loses the original function's `__name__`, `__doc__`, and other metadata. `functools.wraps` copies them.

In [ ]:
import functools

def without_wraps(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def with_wraps(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@without_wraps
def my_func():
    """My docstring."""
    pass

@with_wraps
def my_func2():
    """My docstring."""
    pass

print("Without wraps:", my_func.__name__, my_func.__doc__)
print("With wraps:", my_func2.__name__, my_func2.__doc__)

## Why wraps Matters

Debuggers, help(), and documentation tools rely on `__name__` and `__doc__`. Always use `@functools.wraps(func)` in production decorators.

In [ ]:
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        import time
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
        return result
    return wrapper

@timer
def compute():
    """Compute something."""
    return sum(range(1000))

compute()
print(help(compute))  # Shows correct name and docstring

## Class-Based Decorators

A class can be a decorator if it implements `__call__`. Useful when you need to maintain state.

In [ ]:
import functools

class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        return self.func(*args, **kwargs)

@CountCalls
def say_hello():
    print("Hello")

say_hello()
say_hello()
print(f"Called {say_hello.count} times")

## Decorating Methods

When decorating methods, the first argument is `self`. Your wrapper must pass it through.

In [ ]:
import functools

def log_method(func):
    @functools.wraps(func)
    def wrapper(self, *args, **kwargs):
        print(f"{self.__class__.__name__}.{func.__name__} called")
        return func(self, *args, **kwargs)
    return wrapper

class Calculator:
    @log_method
    def add(self, a, b):
        return a + b

c = Calculator()
print(c.add(2, 3))

## Common Mistakes

- **Forgetting functools.wraps**: Loses __name__, __doc__, breaks help()
- **Wrong decorator order when stacking**: Bottom decorator runs first (innermost)
- **Decorator with optional args**: Use `func is not None` check
- **Mutable default in decorator factory**: Same pitfall as in regular functions

## Exercises

In [ ]:
# Exercise 1: Create a decorator @deprecated(message) that prints the message when the function is called.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Create a decorator that supports both @rate_limit and @rate_limit(per_minute=10).
# YOUR CODE HERE

In [ ]:
# Exercise 3: Stack @timer and @log_calls. Which runs first? Verify with a test function.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Create a class-based decorator that caches results by argument hash.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Create a decorator that validates return type matches an annotation.
# @validate_return
# def get_int() -> int: return 42
# YOUR CODE HERE

In [ ]:
# Exercise 6: Fix a decorator that loses __name__ by adding functools.wraps.
# YOUR CODE HERE

## Key Takeaways

- Decorators with args: factory returns decorator
- Stacking: `@a @b def f` → `a(b(f))`, b runs first
- Always use `@functools.wraps(func)`
- Class decorators: implement `__call__`

## 🔜 Next: Day 3 - Class Decorators

Tomorrow: class decorators and the descriptor protocol!